# Preparación del dataset BOSSBase para entrenamiento en PyTorch sin data leakage

Este notebook genera la carpeta que usa el notebook de entrenamiento en PyTorch:

```text
dataset/
├── cover/
├── stego_7/
└── stego_15/
```

Además, crea manifiestos de partición a nivel de imagen base:

```text
splits/
├── train_base_names.txt
├── val_base_names.txt
└── test_base_names.txt
```

La idea central es que el split se hace sobre el identificador de la imagen fuente de BOSSBase antes de generar o usar pares `cover/stego`. Así, una misma imagen base nunca queda simultáneamente en entrenamiento, validación o prueba.

## 1. Importaciones

Este notebook está pensado para ejecutarse en un `.ipynb` normal, no depende de comandos de Colab como `!wget`. La descarga se realiza con `urllib.request`.

In [1]:
import os
import re
import cv2
import json
import time
import glob
import shutil
import zipfile
import hashlib
import urllib.request
from pathlib import Path

import numpy as np

## 2. Configuración del dataset

Los valores de `VALIDATION_SPLIT`, `TEST_SPLIT` y `SEMILLA` coinciden con el notebook de entrenamiento en PyTorch entregado:

```python
VALIDATION_SPLIT = 0.25
TEST_SPLIT = 0.15
SEMILLA = 42
```

Con `CANTIDAD_A_PROCESAR = 1000`, el split esperado es aproximadamente:

- Train: 600 imágenes base.
- Val: 250 imágenes base.
- Test: 150 imágenes base.

Cada imagen base genera tres archivos asociados dentro de `dataset/`: una cover original, una stego con Hamming (7,4) y una stego con Hamming (15,11).

In [2]:
# ============================================================
# CONFIGURACIÓN GENERAL
# ============================================================

URL_BOSSBASE = "http://dde.binghamton.edu/download/ImageDB/BOSSbase_1.01.zip"

# Carpeta raíz del proyecto.
# En un notebook local, "." significa la carpeta donde está este notebook.
RUTA_PROYECTO = Path(".")

# Archivos y carpetas de entrada
ZIP_PATH = RUTA_PROYECTO / "bossbase.zip"
EXTRACT_DIR = RUTA_PROYECTO / "bossbase_completo"

# Carpetas de salida compatibles con el notebook de entrenamiento en PyTorch
DATASET_DIR = RUTA_PROYECTO / "dataset"
COVER_DIR = DATASET_DIR / "cover"
STEGO_7_DIR = DATASET_DIR / "stego_7"
STEGO_15_DIR = DATASET_DIR / "stego_15"

# Manifiestos de split
SPLIT_DIR = RUTA_PROYECTO / "splits"
TRAIN_SPLIT_PATH = SPLIT_DIR / "train_base_names.txt"
VAL_SPLIT_PATH = SPLIT_DIR / "val_base_names.txt"
TEST_SPLIT_PATH = SPLIT_DIR / "test_base_names.txt"

# Parámetros experimentales
CANTIDAD_A_PROCESAR = 1000
VALIDATION_SPLIT = 0.25
TEST_SPLIT = 0.15
SEMILLA = 42

# Si es True, borra y reconstruye dataset/ y splits/
RECONSTRUIR_DATASET = True

# Formato de las imágenes cover.
# "pgm": conserva el formato original de BOSSBase.
# "png": convierte las cover a PNG sin pérdidas.
FORMATO_COVER = "pgm"

if FORMATO_COVER not in {"pgm", "png"}:
    raise ValueError("FORMATO_COVER debe ser 'pgm' o 'png'.")

print("Configuración lista.")
print(f"Ruta del proyecto: {RUTA_PROYECTO.resolve()}")
print(f"Dataset de salida: {DATASET_DIR.resolve()}")

Configuración lista.
Ruta del proyecto: C:\Users\m.amorocho\proyecto-cripto
Dataset de salida: C:\Users\m.amorocho\proyecto-cripto\dataset


## 3. Funciones de Matrix Embedding

Se incluyen las funciones necesarias para generar las imágenes stego. La generación se hace en formato PNG porque PNG es sin pérdida y no destruye los bits menos significativos modificados por el embedding.

In [3]:
# ============================================================
# MATRICES DE COMPROBACIÓN DE PARIDAD
# ============================================================

H_7_3 = np.array([
    [0, 0, 0, 1, 1, 1, 1],
    [0, 1, 1, 0, 0, 1, 1],
    [1, 0, 1, 0, 1, 0, 1]
], dtype=np.uint8)

H_15_11 = np.array([
    [0, 0, 0, 0, 0, 0, 0, 1, 1, 1, 1, 1, 1, 1, 1],
    [0, 0, 0, 1, 1, 1, 1, 0, 0, 0, 0, 1, 1, 1, 1],
    [0, 1, 1, 0, 0, 1, 1, 0, 0, 1, 1, 0, 0, 1, 1],
    [1, 0, 1, 0, 1, 0, 1, 0, 1, 0, 1, 0, 1, 0, 1]
], dtype=np.uint8)


def embed_hamming_7_3(pixels_block, secret_bits):
    """
    Oculta 3 bits en un bloque de 7 píxeles usando Matrix Embedding.
    Esta función se deja explícita por compatibilidad conceptual con el notebook de entrenamiento.
    """
    x = pixels_block % 2
    s_current = np.dot(H_7_3, x) % 2
    d = (secret_bits - s_current) % 2
    index_to_change = int(d[0] * 4 + d[1] * 2 + d[2])

    modified_pixels = pixels_block.copy()

    if index_to_change > 0:
        idx = index_to_change - 1
        if modified_pixels[idx] % 2 == 0:
            modified_pixels[idx] += 1
        else:
            modified_pixels[idx] -= 1

    return modified_pixels


def embed_hamming_15_11(pixels_block, secret_bits):
    """
    Oculta 4 bits en un bloque de 15 píxeles usando Matrix Embedding.
    """
    x = pixels_block % 2
    s_current = np.dot(H_15_11, x) % 2
    d = (secret_bits - s_current) % 2
    index_to_change = int(d[0] * 8 + d[1] * 4 + d[2] * 2 + d[3])

    modified_pixels = pixels_block.copy()

    if index_to_change > 0:
        idx = index_to_change - 1
        if modified_pixels[idx] % 2 == 0:
            modified_pixels[idx] += 1
        else:
            modified_pixels[idx] -= 1

    return modified_pixels


def crear_rng_por_imagen(base_name, esquema, seed_global=42):
    """
    Crea un generador pseudoaleatorio determinístico por imagen y esquema.
    Esto garantiza reproducibilidad aunque cambie el orden de procesamiento.
    """
    clave = f"{seed_global}|{esquema}|{base_name}".encode("utf-8")
    digest = hashlib.sha256(clave).digest()
    seed = int.from_bytes(digest[:4], byteorder="little", signed=False)
    return np.random.default_rng(seed)


def aplicar_matrix_embedding_vectorizado(img, n, k_bits, H, pesos_binarios, rng):
    """
    Aplica Matrix Embedding a una imagen completa de forma vectorizada.

    Parámetros:
    - img: imagen 2D uint8 en escala de grises.
    - n: tamaño del bloque de píxeles, 7 o 15.
    - k_bits: cantidad de bits ocultos por bloque, 3 o 4.
    - H: matriz de comprobación de paridad.
    - pesos_binarios: pesos para convertir el síndrome a índice decimal.
    - rng: generador aleatorio de NumPy.

    Retorna:
    - Imagen stego 2D uint8.
    """
    if img.ndim != 2:
        raise ValueError("La imagen debe estar en escala de grises con forma (alto, ancho).")

    flat = img.reshape(-1).astype(np.uint8, copy=True)

    num_blocks = len(flat) // n
    if num_blocks == 0:
        return flat.reshape(img.shape)

    usable = num_blocks * n
    blocks = flat[:usable].reshape(num_blocks, n)

    secret = rng.integers(0, 2, size=(num_blocks, k_bits), dtype=np.uint8)

    lsb = blocks % 2
    syndrome = (lsb @ H.T) % 2
    d = (secret - syndrome) % 2

    indices = (d @ pesos_binarios).astype(np.int16)
    mask = indices > 0

    if np.any(mask):
        filas = np.where(mask)[0]
        columnas = indices[mask] - 1

        valores = blocks[filas, columnas].astype(np.int16)

        # Cambiar LSB sin salir de [0, 255].
        nuevos_valores = np.where(valores % 2 == 0, valores + 1, valores - 1)
        blocks[filas, columnas] = nuevos_valores.astype(np.uint8)

    return flat.reshape(img.shape)


def generar_stego_7(img, base_name, seed_global=42):
    rng = crear_rng_por_imagen(base_name, "stego_7", seed_global)
    return aplicar_matrix_embedding_vectorizado(
        img=img,
        n=7,
        k_bits=3,
        H=H_7_3,
        pesos_binarios=np.array([4, 2, 1], dtype=np.uint8),
        rng=rng
    )


def generar_stego_15(img, base_name, seed_global=42):
    rng = crear_rng_por_imagen(base_name, "stego_15", seed_global)
    return aplicar_matrix_embedding_vectorizado(
        img=img,
        n=15,
        k_bits=4,
        H=H_15_11,
        pesos_binarios=np.array([8, 4, 2, 1], dtype=np.uint8),
        rng=rng
    )

## 4. Funciones de descarga, extracción y split por imagen base

La función de split replica la lógica del notebook de entrenamiento en PyTorch: primero toma los stems comunes, los mezcla con `SEMILLA = 42`, separa test, luego validación y finalmente entrenamiento.

In [4]:
# ============================================================
# UTILIDADES DE DESCARGA Y PREPARACIÓN
# ============================================================

def natural_key(path):
    """
    Orden natural para seleccionar las primeras N imágenes de manera intuitiva:
    1, 2, 3, ..., 10, 11, ...
    """
    nombre = Path(path).name
    return [int(t) if t.isdigit() else t.lower() for t in re.split(r"(\d+)", nombre)]


def descargar_con_progreso(url, destino):
    """
    Descarga un archivo usando urllib, compatible con notebooks locales.
    """
    destino = Path(destino)

    def hook(bloques, tam_bloque, tam_total):
        if tam_total > 0:
            descargado = bloques * tam_bloque
            porcentaje = min(100, descargado * 100 / tam_total)
            print(f"\rDescargando: {porcentaje:6.2f}%", end="")

    urllib.request.urlretrieve(url, destino, reporthook=hook)
    print("\nDescarga finalizada.")


def extraer_zip(zip_path, extract_dir):
    zip_path = Path(zip_path)
    extract_dir = Path(extract_dir)

    print("Extrayendo archivo ZIP...")
    with zipfile.ZipFile(zip_path, "r") as zip_ref:
        zip_ref.extractall(extract_dir)

    print("Extracción finalizada.")


def localizar_imagenes_bossbase(extract_dir):
    """
    Localiza todas las imágenes .pgm dentro de la carpeta extraída.
    Se usa rglob para tolerar variaciones en la estructura interna del ZIP.
    """
    extract_dir = Path(extract_dir)

    rutas = [
        p for p in extract_dir.rglob("*.pgm")
        if "__MACOSX" not in str(p)
    ]

    rutas = sorted(rutas, key=natural_key)

    if len(rutas) == 0:
        raise FileNotFoundError(
            f"No se encontraron imágenes .pgm dentro de {extract_dir.resolve()}."
        )

    return rutas


def dividir_stems_sin_solapamiento(stems, validation_split=0.25, test_split=0.15, seed=42):
    """
    Divide identificadores de imagen base sin solapamiento entre train, val y test.

    Esta función replica la lógica del notebook de entrenamiento en PyTorch.
    """
    rng = np.random.default_rng(seed)
    stems = np.array(sorted(stems))
    rng.shuffle(stems)

    n_total = len(stems)
    if n_total < 3:
        raise ValueError("Se necesitan al menos 3 imágenes base para crear train, val y test.")

    n_test = max(1, int(round(n_total * test_split))) if test_split > 0 else 0
    n_val = max(1, int(round(n_total * validation_split))) if validation_split > 0 else 0

    if n_test + n_val >= n_total:
        n_test = max(1, min(n_test, n_total - 2))
        n_val = max(1, min(n_val, n_total - n_test - 1))

    stems_test = stems[:n_test]
    stems_val = stems[n_test:n_test + n_val]
    stems_train = stems[n_test + n_val:]

    assert len(set(stems_train) & set(stems_val)) == 0
    assert len(set(stems_train) & set(stems_test)) == 0
    assert len(set(stems_val) & set(stems_test)) == 0

    return list(stems_train), list(stems_val), list(stems_test)


def guardar_lista(path, valores):
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    path.write_text("\n".join(map(str, valores)), encoding="utf-8")


def leer_lista(path):
    path = Path(path)
    return [line.strip() for line in path.read_text(encoding="utf-8").splitlines() if line.strip()]

## 5. Descarga y extracción de BOSSBase 1.01

Esta celda descarga el ZIP oficial solo si no existe previamente. Si ya existe `bossbase.zip`, se reutiliza.

In [5]:
# ============================================================
# DESCARGA Y EXTRACCIÓN
# ============================================================

if not ZIP_PATH.exists():
    print("Descargando BOSSBase 1.01 desde el repositorio oficial...")
    descargar_con_progreso(URL_BOSSBASE, ZIP_PATH)
else:
    print(f"El archivo {ZIP_PATH} ya existe. Se omite la descarga.")

# Extraer si la carpeta no existe o si no contiene imágenes .pgm.
debe_extraer = True
if EXTRACT_DIR.exists():
    try:
        rutas_existentes = localizar_imagenes_bossbase(EXTRACT_DIR)
        debe_extraer = len(rutas_existentes) == 0
    except FileNotFoundError:
        debe_extraer = True

if debe_extraer:
    EXTRACT_DIR.mkdir(parents=True, exist_ok=True)
    extraer_zip(ZIP_PATH, EXTRACT_DIR)
else:
    print(f"La carpeta {EXTRACT_DIR} ya contiene imágenes .pgm. Se omite la extracción.")

rutas_bossbase = localizar_imagenes_bossbase(EXTRACT_DIR)
print(f"Imágenes .pgm encontradas: {len(rutas_bossbase)}")

if CANTIDAD_A_PROCESAR > len(rutas_bossbase):
    raise ValueError(
        f"CANTIDAD_A_PROCESAR={CANTIDAD_A_PROCESAR}, "
        f"pero solo hay {len(rutas_bossbase)} imágenes disponibles."
    )

rutas_seleccionadas = rutas_bossbase[:CANTIDAD_A_PROCESAR]
base_names = [p.stem for p in rutas_seleccionadas]

if len(base_names) != len(set(base_names)):
    raise ValueError("Hay nombres base duplicados en las imágenes seleccionadas.")

print(f"Imágenes seleccionadas para el experimento: {len(rutas_seleccionadas)}")
print(f"Primer stem: {base_names[0]} | Último stem seleccionado: {base_names[-1]}")

Descargando BOSSBase 1.01 desde el repositorio oficial...
Descargando: 100.00%
Descarga finalizada.
Extrayendo archivo ZIP...
Extracción finalizada.
Imágenes .pgm encontradas: 10000
Imágenes seleccionadas para el experimento: 1000
Primer stem: 1 | Último stem seleccionado: 1000


## 6. Split por imagen base

El split se hace antes de construir los pares `cover/stego`. Esto evita que una imagen base aparezca en más de una partición.

Aunque las imágenes se guardan físicamente en carpetas por clase (`cover`, `stego_7`, `stego_15`), los archivos `.txt` de `splits/` documentan qué stems pertenecen a train, validation y test.

In [6]:
# ============================================================
# SPLIT SIN DATA LEAKAGE
# ============================================================

stems_train, stems_val, stems_test = dividir_stems_sin_solapamiento(
    stems=base_names,
    validation_split=VALIDATION_SPLIT,
    test_split=TEST_SPLIT,
    seed=SEMILLA
)

train_set = set(stems_train)
val_set = set(stems_val)
test_set = set(stems_test)

assert train_set.isdisjoint(val_set)
assert train_set.isdisjoint(test_set)
assert val_set.isdisjoint(test_set)

SPLIT_DIR.mkdir(parents=True, exist_ok=True)

guardar_lista(TRAIN_SPLIT_PATH, stems_train)
guardar_lista(VAL_SPLIT_PATH, stems_val)
guardar_lista(TEST_SPLIT_PATH, stems_test)

print("Split por imagen base creado correctamente:")
print(f"Train: {len(stems_train)} imágenes base")
print(f"Val:   {len(stems_val)} imágenes base")
print(f"Test:  {len(stems_test)} imágenes base")
print("Solapamiento train/val/test: 0")

print("\nManifiestos guardados en:")
print(TRAIN_SPLIT_PATH)
print(VAL_SPLIT_PATH)
print(TEST_SPLIT_PATH)

Split por imagen base creado correctamente:
Train: 600 imágenes base
Val:   250 imágenes base
Test:  150 imágenes base
Solapamiento train/val/test: 0

Manifiestos guardados en:
splits\train_base_names.txt
splits\val_base_names.txt
splits\test_base_names.txt


## 7. Generación de la carpeta `dataset/`

Esta celda crea la estructura final esperada por el notebook de entrenamiento:

```text
dataset/
├── cover/
├── stego_7/
└── stego_15/
```

La carpeta `cover/` contiene las imágenes originales de BOSSBase. Las carpetas `stego_7/` y `stego_15/` contienen las versiones modificadas mediante Matrix Embedding.

In [7]:
# ============================================================
# GENERACIÓN DEL DATASET CLASIFICADO PARA PYTORCH
# ============================================================

if RECONSTRUIR_DATASET:
    if DATASET_DIR.exists():
        shutil.rmtree(DATASET_DIR)

COVER_DIR.mkdir(parents=True, exist_ok=True)
STEGO_7_DIR.mkdir(parents=True, exist_ok=True)
STEGO_15_DIR.mkdir(parents=True, exist_ok=True)

rutas_por_stem = {p.stem: p for p in rutas_seleccionadas}

# Procesar por partición para respetar conceptualmente el split base.
# Las salidas se guardan en carpetas por clase porque así las usa el notebook de entrenamiento.
particiones = [
    ("train", stems_train),
    ("val", stems_val),
    ("test", stems_test),
]

inicio = time.time()
total_procesadas = 0
total_a_procesar = sum(len(stems) for _, stems in particiones)

print("Iniciando generación del dataset clasificado...")
print(f"Total de imágenes base a procesar: {total_a_procesar}")

for nombre_particion, stems_particion in particiones:
    print(f"\nProcesando partición lógica: {nombre_particion} ({len(stems_particion)} imágenes base)")

    for i, stem in enumerate(stems_particion, start=1):
        src_path = rutas_por_stem[stem]

        # 1. Copiar o convertir cover
        img = cv2.imread(str(src_path), cv2.IMREAD_GRAYSCALE)
        if img is None:
            raise ValueError(f"No se pudo leer la imagen fuente: {src_path}")

        if FORMATO_COVER == "pgm":
            cover_out = COVER_DIR / f"{stem}.pgm"
            shutil.copy2(src_path, cover_out)
        else:
            cover_out = COVER_DIR / f"{stem}.png"
            cv2.imwrite(str(cover_out), img)

        # 2. Generar stego Hamming (7,4)
        stego_7 = generar_stego_7(img, base_name=stem, seed_global=SEMILLA)
        stego_7_out = STEGO_7_DIR / f"{stem}.png"
        cv2.imwrite(str(stego_7_out), stego_7)

        # 3. Generar stego Hamming (15,11)
        stego_15 = generar_stego_15(img, base_name=stem, seed_global=SEMILLA)
        stego_15_out = STEGO_15_DIR / f"{stem}.png"
        cv2.imwrite(str(stego_15_out), stego_15)

        total_procesadas += 1

        if i % 100 == 0 or i == len(stems_particion):
            print(
                f"  {nombre_particion}: {i}/{len(stems_particion)} "
                f"| total: {total_procesadas}/{total_a_procesar}"
            )

duracion = time.time() - inicio

print("\nGeneración finalizada.")
print(f"Tiempo total: {duracion / 60:.2f} minutos")

Iniciando generación del dataset clasificado...
Total de imágenes base a procesar: 1000

Procesando partición lógica: train (600 imágenes base)
  train: 100/600 | total: 100/1000
  train: 200/600 | total: 200/1000
  train: 300/600 | total: 300/1000
  train: 400/600 | total: 400/1000
  train: 500/600 | total: 500/1000
  train: 600/600 | total: 600/1000

Procesando partición lógica: val (250 imágenes base)
  val: 100/250 | total: 700/1000
  val: 200/250 | total: 800/1000
  val: 250/250 | total: 850/1000

Procesando partición lógica: test (150 imágenes base)
  test: 100/150 | total: 950/1000
  test: 150/150 | total: 1000/1000

Generación finalizada.
Tiempo total: 0.19 minutos


## 8. Verificación final

Esta celda verifica:

1. Que existan las tres carpetas esperadas.
2. Que las tres clases tengan la misma cantidad de imágenes.
3. Que cada `cover` tenga un par con el mismo stem en `stego_7` y `stego_15`.
4. Que no exista solapamiento entre train, val y test a nivel de imagen base.

In [8]:
# ============================================================
# VERIFICACIÓN FINAL DEL DATASET
# ============================================================

EXTENSIONES_IMG = ["*.png", "*.jpg", "*.jpeg", "*.bmp", "*.pgm", "*.tif", "*.tiff"]

def listar_stems(carpeta):
    stems = set()
    for ext in EXTENSIONES_IMG:
        for ruta in Path(carpeta).glob(ext):
            stems.add(ruta.stem)
    return stems


cover_stems = listar_stems(COVER_DIR)
stego_7_stems = listar_stems(STEGO_7_DIR)
stego_15_stems = listar_stems(STEGO_15_DIR)

print("Conteo final:")
print(f"dataset/cover:    {len(cover_stems)} imágenes")
print(f"dataset/stego_7:  {len(stego_7_stems)} imágenes")
print(f"dataset/stego_15: {len(stego_15_stems)} imágenes")

assert len(cover_stems) == CANTIDAD_A_PROCESAR
assert cover_stems == stego_7_stems, "No todos los cover tienen par en stego_7."
assert cover_stems == stego_15_stems, "No todos los cover tienen par en stego_15."

stems_train_check = set(leer_lista(TRAIN_SPLIT_PATH))
stems_val_check = set(leer_lista(VAL_SPLIT_PATH))
stems_test_check = set(leer_lista(TEST_SPLIT_PATH))

assert stems_train_check.isdisjoint(stems_val_check)
assert stems_train_check.isdisjoint(stems_test_check)
assert stems_val_check.isdisjoint(stems_test_check)

assert stems_train_check | stems_val_check | stems_test_check == cover_stems

print("\nVerificación anti-leakage correcta:")
print(" - No hay imágenes base repetidas entre train, val y test.")
print(" - Los stems de cover, stego_7 y stego_15 coinciden exactamente.")
print(" - La carpeta dataset/ quedó lista para el notebook de entrenamiento en PyTorch.")

print("\nEstructura generada:")
print("dataset/")
print("├── cover/")
print("├── stego_7/")
print("└── stego_15/")
print("\nsplits/")
print("├── train_base_names.txt")
print("├── val_base_names.txt")
print("└── test_base_names.txt")

Conteo final:
dataset/cover:    1000 imágenes
dataset/stego_7:  1000 imágenes
dataset/stego_15: 1000 imágenes

Verificación anti-leakage correcta:
 - No hay imágenes base repetidas entre train, val y test.
 - Los stems de cover, stego_7 y stego_15 coinciden exactamente.
 - La carpeta dataset/ quedó lista para el notebook de entrenamiento en PyTorch.

Estructura generada:
dataset/
├── cover/
├── stego_7/
└── stego_15/

splits/
├── train_base_names.txt
├── val_base_names.txt
└── test_base_names.txt


## 9. Metadata del dataset

Se guarda un archivo JSON con los parámetros usados para que el experimento sea reproducible.

In [9]:
# ============================================================
# GUARDAR METADATA DEL DATASET
# ============================================================

metadata = {
    "dataset_dir": str(DATASET_DIR.resolve()),
    "bossbase_url": URL_BOSSBASE,
    "cantidad_a_procesar": CANTIDAD_A_PROCESAR,
    "validation_split": VALIDATION_SPLIT,
    "test_split": TEST_SPLIT,
    "train_base_images": len(stems_train),
    "val_base_images": len(stems_val),
    "test_base_images": len(stems_test),
    "seed": SEMILLA,
    "cover_format": FORMATO_COVER,
    "output_structure": {
        "cover": str(COVER_DIR),
        "stego_7": str(STEGO_7_DIR),
        "stego_15": str(STEGO_15_DIR),
        "splits": str(SPLIT_DIR),
    },
    "anti_leakage_rule": (
        "El split se realiza sobre el identificador de imagen base antes de usar los pares cover/stego."
    )
}

metadata_path = RUTA_PROYECTO / "dataset_metadata_pytorch.json"
metadata_path.write_text(json.dumps(metadata, indent=4, ensure_ascii=False), encoding="utf-8")

print(f"Metadata guardada en: {metadata_path}")
print(json.dumps(metadata, indent=4, ensure_ascii=False))

Metadata guardada en: dataset_metadata_pytorch.json
{
    "dataset_dir": "C:\\Users\\m.amorocho\\proyecto-cripto\\dataset",
    "bossbase_url": "http://dde.binghamton.edu/download/ImageDB/BOSSbase_1.01.zip",
    "cantidad_a_procesar": 1000,
    "validation_split": 0.25,
    "test_split": 0.15,
    "train_base_images": 600,
    "val_base_images": 250,
    "test_base_images": 150,
    "seed": 42,
    "cover_format": "pgm",
    "output_structure": {
        "cover": "dataset\\cover",
        "stego_7": "dataset\\stego_7",
        "stego_15": "dataset\\stego_15",
        "splits": "splits"
    },
    "anti_leakage_rule": "El split se realiza sobre el identificador de imagen base antes de usar los pares cover/stego."
}


## 10. Uso con el notebook de entrenamiento

Después de ejecutar este notebook, el notebook `hamming_embedding_resnet_steganalysis.ipynb` puede usar la carpeta generada directamente.

Si ambos notebooks están en la misma carpeta, basta con dejar:

```python
RUTA_PROYECTO = "."
```

El entrenamiento leerá:

```python
dataset/cover
dataset/stego_7
dataset/stego_15
```

y su función `crear_dataloaders_binarios()` volverá a dividir por `stem`, no por archivo individual, evitando data leakage.